# 数据科学 100 道练习 31~40

In [1]:
import polars as pl
from helper.polars import load_100knocks_data
pl.Config.set_fmt_str_lengths(100)
df_customer, df_category, df_product, df_receipt, df_store, df_geocode = load_100knocks_data()

## P-031

对收据明细数据（df_receipt），按店铺代码（store_cd）计算销售金额（amount）的标准差，按降序显示 5 条。

In [2]:
(
df_receipt
.group_by('store_cd')
.agg(
    std_amount=pl.col.amount.std(ddof=0)
)
.sort(by='std_amount', descending=True)
.head()
)

store_cd,std_amount
str,f64
"""S13052""",663.391816
"""S14011""",553.456916
"""S14034""",544.903736
"""S13001""",543.536561
"""S13015""",543.409938


## P-032

求收据明细数据（df_receipt）中销售金额（amount）按 25% 间隔划分的百分位值。

In [3]:
df_receipt.select(
    [pl.col.amount
     .quantile(per / 100)
     .alias(f'amount_{per}per') 
    for per in [25, 50, 75, 100]
    ]
)

amount_25per,amount_50per,amount_75per,amount_100per
f64,f64,f64,f64
102.0,170.0,288.0,10925.0


## P-033

对收据明细数据（df_receipt），按店铺代码（store_cd）计算销售金额（amount）的平均值，抽取平均值在 330 以上的记录。

In [4]:
(
df_receipt
.group_by('store_cd', maintain_order=True)
.agg(
    avg_amount=pl.col.amount.mean()
)
.filter(pl.col.avg_amount >= 330)
)

store_cd,avg_amount
str,f64
"""S13003""",350.915519
"""S13020""",337.879932
"""S14026""",332.340588
"""S13015""",351.11196
"""S14045""",330.082073
…,…
"""S14010""",348.791262
"""S12013""",330.19413
"""S13001""",348.470386


## P-034

对收据明细数据（df_receipt），按顾客 ID（customer_id）合计销售金额（amount），再求全部顾客的平均值。顾客 ID 以 "Z" 开头的是非会员，计算时排除在外。

In [5]:
(
df_receipt
.filter(
    pl.col.customer_id.str.starts_with('Z').not_()
)
.group_by('customer_id')
.agg(
    pl.col.amount.sum()
)
.select(pl.col.amount.mean())
.item()
)

2547.742234529256

## P-035

对收据明细数据（df_receipt），按顾客 ID（customer_id）合计销售金额（amount），求全部顾客的平均值，抽取消费金额在平均值以上的顾客，显示 10 条。顾客 ID 以 "Z" 开头的是非会员，计算时排除在外。

In [6]:
(
df_receipt
.lazy()
.filter(
    pl.col.customer_id.str.starts_with('Z').not_()
)
.group_by('customer_id')
.agg(
    sum_amount = pl.col.amount.sum()
)
.filter(
    pl.col.sum_amount >= pl.col.sum_amount.mean()
)
.collect()
.head(10)
)

customer_id,sum_amount
str,i64
"""CS015415000003""",10744
"""CS039515000204""",4886
"""CS019415000147""",12764
"""CS010514000041""",7623
"""CS034605000007""",4694
"""CS005214000014""",3409
"""CS021615000133""",3455
"""CS027515000065""",3171
"""CS026415000256""",4081


## P-036

把收据明细数据（df_receipt）与店铺数据（df_store）做内连接，显示收据明细数据的全部字段和店铺数据的店铺名（store_name），共 10 条。

In [7]:
(
df_receipt
.join(
    df_store.select('store_cd', 'store_name'), 
    on='store_cd'
)
.head(10)
)

sales_ymd,sales_epoch,store_cd,receipt_no,receipt_sub_no,customer_id,product_cd,quantity,amount,store_name
i64,i64,str,i64,i64,str,str,i64,i64,str
20181103,1541203200,"""S14006""",112,1,"""CS006214000001""","""P070305012""",1,158,"""葛が谷店"""
20181118,1542499200,"""S13008""",1132,2,"""CS008415000097""","""P070701017""",1,81,"""成城店"""
20170712,1499817600,"""S14028""",1102,1,"""CS028414000014""","""P060101005""",1,170,"""二ツ橋店"""
20190205,1549324800,"""S14042""",1132,1,"""ZZ000000000000""","""P050301001""",1,25,"""新山下店"""
20180821,1534809600,"""S14025""",1102,2,"""CS025415000050""","""P060102007""",1,90,"""大和店"""
20190605,1559692800,"""S13003""",1112,1,"""CS003515000195""","""P050102002""",1,138,"""狛江店"""
20181205,1543968000,"""S14024""",1102,2,"""CS024514000042""","""P080101005""",1,30,"""三田店"""
20190922,1569110400,"""S14040""",1102,1,"""CS040415000178""","""P070501004""",1,128,"""長津田店"""
20170504,1493856000,"""S13020""",1112,2,"""ZZ000000000000""","""P071302010""",1,770,"""十条仲原店"""


## P-037

把商品数据（df_product）与分类数据（df_category）做内连接，显示商品数据的全部字段和分类数据的分类小类名（category_small_name），共 10 条。

In [8]:
(
df_product
.join(
    df_category.select(
        'category_small_cd', 
        'category_small_name'
    ),
    on='category_small_cd'
)
.head(10)
)

product_cd,category_major_cd,category_medium_cd,category_small_cd,unit_price,unit_cost,category_small_name
str,str,str,str,i64,i64,str
"""P040101001""","""04""","""0401""","""040101""",198,149,"""弁当類"""
"""P040101002""","""04""","""0401""","""040101""",218,164,"""弁当類"""
"""P040101003""","""04""","""0401""","""040101""",230,173,"""弁当類"""
"""P040101004""","""04""","""0401""","""040101""",248,186,"""弁当類"""
"""P040101005""","""04""","""0401""","""040101""",268,201,"""弁当類"""
"""P040101006""","""04""","""0401""","""040101""",298,224,"""弁当類"""
"""P040101007""","""04""","""0401""","""040101""",338,254,"""弁当類"""
"""P040101008""","""04""","""0401""","""040101""",420,315,"""弁当類"""
"""P040101009""","""04""","""0401""","""040101""",498,374,"""弁当類"""


## P-038

从顾客数据（df_customer）和收据明细数据（df_receipt）求每位顾客的销售金额合计，显示 10 条。没有销售实绩的顾客，销售金额显示为 0。顾客只取性别代码（gender_cd）为女性（1）的，非会员（顾客 ID 以 "Z" 开头）排除在外。

In [9]:
(
df_customer
.lazy()
.filter(
    (pl.col.gender_cd == "1") & 
    (pl.col.customer_id.str.starts_with('Z').not_())
)
.select('customer_id')
.join(
    df_receipt
    .lazy()
    .group_by('customer_id')
    .agg(
        sum_amount = pl.col.amount.sum()
    ),
    on='customer_id',
    how='left'
)
.with_columns(
    pl.col.sum_amount.fill_null(0)
)
.head(10)
.collect()
)

customer_id,sum_amount
str,i64
"""CS040412000191""",210
"""CS011215000048""",3444
"""CS031415000172""",5088
"""CS015414000103""",3122
"""CS033513000180""",868
"""CS001215000145""",875
"""CS028811000001""",0
"""CS009413000079""",0
"""CS035614000014""",0


## P-039

从收据明细数据（df_receipt）分别创建两份数据：销售天数多的前 20 位顾客、销售金额合计高的前 20 位顾客，再把两者做全外连接。非会员（顾客 ID 以 "Z" 开头）排除在外。

In [10]:
df_data = df_receipt.filter(
    pl.col.customer_id.str.starts_with('Z').not_()
)

group = df_data.group_by('customer_id')

df_cnt = (
    group
    .agg(
        come_days = pl.col.sales_ymd.n_unique()
    )
    .top_k(20, by='come_days')
)

df_sum = (
    group
    .agg(
        buy_amount = pl.col.amount.sum()
    )
    .top_k(20, by='buy_amount')
)

df_cnt.join(df_sum, on='customer_id', how='full', coalesce=True)

customer_id,come_days,buy_amount
str,u32,i64
"""CS017415000097""",20,23086
"""CS015415000185""",22,20153
"""CS031414000051""",19,19202
"""CS028415000007""",21,19127
"""CS001605000009""",null,18925
…,…,…
"""CS022515000226""",19,null
"""CS021515000172""",19,null
"""CS031414000073""",18,null


## P-040

想创建所有店铺与所有商品的组合数据。把店铺数据（df_store）与商品数据（df_product）做笛卡尔积，统计条数。

In [11]:
(
df_store
.join(df_product, how='cross')
.select(pl.len())
.item()
)

531590